# Adding Labor Market Information (LMI) to the CareerNet Dataset

This notebook guides you through the process of obtaining [Labor Market Information (LMI) from the Bureau of Labor Statistics (BLS)](https://www.bls.gov/oes/tables.htm) and integrating it into the [CareerNet dataset](https://github.com/RenaissancePhilanthropy/careernet-data).

By completing this process, you will enrich the CareerNet data with real-world employment figures and salary expectations by joining both datasets using their Standard Occupational Classification (SOC) codes.

*Note: This notebook focuses on total employment, annual mean wage, annual median wage, and the 25th and 75th percentile wages. The code can be adjusted to incorporate other BLS LMI data of interest.*

In [ ]:
# Install required packages if they are not already installed in this environment
%pip install -q pandas numpy openpyxl

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import warnings

# Suppress openpyxl warnings for default styles
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

## Step 1: Obtain the LMI Dataset from the BLS

The Bureau of Labor Statistics (BLS) provides Occupational Employment and Wage Statistics (OES). Because the BLS frequently blocks automated Python downloads (HTTP 403 errors), it is best to download this file manually.

1. Navigate to the BLS OES Tables page: [https://www.bls.gov/oes/tables.htm](https://www.bls.gov/oes/tables.htm)
2. Look for the most recent **National** or **State** data (e.g., "May 2025 National XLSX").
3. Download the Excel file (typically named `national_M2025_dl.xlsx` or similar once downloaded).
4. Place the downloaded `.xlsx` file in the same directory as this Jupyter Notebook.

*Running in Google Colab?* "The same directory as this notebook" means Colab's default `/content` folder. Open the **Files** panel (folder icon in the left sidebar) and upload the file(s) there. Uploaded files are deleted when the runtime resets, so you will need to upload them again in each new session.

*Note: Update the below variables to match your files and data of interest*
- `BLS_DATA_LEVEL` - national or state LMI
- `TARGET_STATES` - if state, which states
- `BLS_YEAR_SUFFIX` - year/name of BLS file being used

*The BLS file is automatically filtered to cross-industry aggregate rows, which represent totals across all industries for each occupation. This leaves one LMI record per SOC code in the national file, and one per SOC code per state in the state file.*

In [ ]:
# ==============================================================================
# CONFIGURATION: BLS Data Settings
# ==============================================================================
# 1. Specify if you are using 'national' or 'state' level data
BLS_DATA_LEVEL = 'state'

# 2. If 'state', list the states you want to keep. Use two-letter state codes.
#    If you want to keep ALL states, leave the list empty: []
TARGET_STATES = ["WA", "AR"]

# 3. (Optional) Update this suffix if you download a different year from the BLS
BLS_YEAR_SUFFIX = 'M2025_dl.xlsx'
# ==============================================================================

# BLS data level guard
assert BLS_DATA_LEVEL.lower() in ('national', 'state'), \
    f"BLS_DATA_LEVEL must be 'national' or 'state', got '{BLS_DATA_LEVEL}'"

# Automatically construct the filename based on the selected level
bls_file_path = f"{BLS_DATA_LEVEL.lower()}_{BLS_YEAR_SUFFIX}"

print(f"Attempting to load BLS dataset from {bls_file_path}...")

try:
    bls_df = pd.read_excel(bls_file_path)
    print("✅ BLS data loaded successfully!")

    if BLS_DATA_LEVEL.lower() == 'state':
        if 'PRIM_STATE' not in bls_df.columns:
            print("❌ Error: 'PRIM_STATE' column not found. Are you sure you downloaded the state-level file?")
        elif TARGET_STATES:
            # Filter the dataframe to only include the requested states
            original_len = len(bls_df)
            bls_df = bls_df[bls_df['PRIM_STATE'].isin(TARGET_STATES)].copy()
            print(f"Filtered to states: {TARGET_STATES}. (Kept {len(bls_df)} out of {original_len} rows)")
        else:
            print("No target states provided. Retaining all 50 states and territories.")

    # Filter to cross-industry aggregate rows to prevent fan-out during the SOC code merge
    pre_filter_len = len(bls_df)
    bls_df = bls_df[bls_df['I_GROUP'] == 'cross-industry'].copy()
    print(f"Filtered to cross-industry rows: retained {len(bls_df)} of {pre_filter_len} rows.")

    display(bls_df.head(3))
except FileNotFoundError:
    print(f"❌ File Not Found: Could not find '{bls_file_path}'.")
    print("   Please ensure the Excel file is in the exact same folder as this notebook.")
except Exception as e:
    print(f"❌ Error reading BLS file: {e}")

## Step 2: Organize and Handle the BLS Excel Data

The BLS spreadsheet contains many data columns, but we primarily care about the LMI statistics that provide career context:
* `OCC_CODE`: The SOC Code (our join key)
* `OCC_TITLE`: The official occupation title
* `TOT_EMP`: Total employment for the occupation
* `A_MEAN`: Annual mean wage
* `A_MEDIAN`: Annual median wage
* `A_PCT25`: Annual 25th percentile wage
* `A_PCT75`: Annual 75th percentile wage

We will extract these columns, handle missing values (the BLS often uses `'*'` or `'#'` for missing or suppressed data), and convert the wage/employment data to numeric types.

*Notes:*
- Update the below variables to match your data of interest
    * `LMI_COLUMN_MAPPING` - BLS LMI variables of interest
    * `NUMERIC_BLS_COLS` - BLS LMI variables of interest that are numeric
- The national file includes major, minor, broad and detailed codes. The state file includes only major and detailed codes, so CareerNet's minor and broad codes (e.g. `29-1000`, `29-1210`) receive LMI data only with national data.

In [ ]:
# ==============================================================================
# CONFIGURATION: Define your LMI Columns here
# ==============================================================================
LMI_COLUMN_MAPPING = {
    'OCC_CODE': 'soc_code',
    'OCC_TITLE': 'soc_title',
    'TOT_EMP': 'total_employment',
    'A_MEAN': 'annual_mean_wage',
    'A_MEDIAN': 'annual_median_wage',
    'A_PCT25': 'annual_25percent_wage',
    'A_PCT75': 'annual_75percent_wage'
}

# Automatically add the state code column to our mapping if we are processing state data
if BLS_DATA_LEVEL.lower() == 'state':
    LMI_COLUMN_MAPPING['PRIM_STATE'] = 'state_code'

NUMERIC_BLS_COLS = ['TOT_EMP', 'A_MEAN', 'A_MEDIAN', 'A_PCT25', 'A_PCT75']
# ==============================================================================

lmi_columns = list(LMI_COLUMN_MAPPING.keys())

# Check if these columns exist to avoid KeyError
missing_cols = [col for col in lmi_columns if col not in bls_df.columns]
if missing_cols:
    print(f"Warning: The following columns are missing from the BLS data: {missing_cols}")
else:
    lmi_df = bls_df[lmi_columns].copy()

    # Define the symbol mapping based on BLS documentation
    symbol_map = {
        '*': 'wage estimate is not available',
        '**': 'employment estimate is not available',
        '#': 'wage equal to or greater than $115.00 per hour or $239,200 per year',
        '~': 'the percent of establishments reporting the occupation is less than 0.5%'
    }

    # Process each numeric column to map symbols to text, while keeping valid numbers as floats
    for col in NUMERIC_BLS_COLS:
        if col in lmi_df.columns:
            lmi_df[col] = lmi_df[col].astype(str).str.strip()
            lmi_df[col + '_note'] = lmi_df[col].map(symbol_map)
            lmi_df[col] = pd.to_numeric(lmi_df[col], errors='coerce')


    # Rename columns based on the dynamic mapping dictionary
    lmi_df.rename(columns=LMI_COLUMN_MAPPING, inplace=True)

    # Rename _note columns to match their renamed base column
    note_col_rename = {
        f'{old_col}_note': f'{new_col}_note'
        for old_col, new_col in LMI_COLUMN_MAPPING.items()
        if old_col in NUMERIC_BLS_COLS
    }
    lmi_df.rename(columns=note_col_rename, inplace=True)

    print("LMI Data organized! BLS symbols replaced with string notes directly in their respective columns:")

    # Dynamically find the new names of the numeric columns to check for notes
    note_cols = [col + '_note' for col in NUMERIC_BLS_COLS if (col + '_note') in lmi_df.columns]

    note_mask = pd.Series(False, index=lmi_df.index)
    for col in note_cols:
        note_mask = note_mask | lmi_df[col].notna()


    rows_with_notes = lmi_df[note_mask]

    if not rows_with_notes.empty:
        display(rows_with_notes.head(3))
    else:
        display(lmi_df.head(3))

## Step 3: Load the CareerNet Dataset

Next, we load the CareerNet dataset. CareerNet contains multiple CSVs across different domains (General, Technology, Allied Health). This code points directly to the raw CSV URL on GitHub to obtain the latest CareerNet data.

*Note: Update the below variables to match your data of interest*
- `SELECTED_DATASETS` - the CareerNet domains of interest

In [ ]:
# ==============================================================================
# CONFIGURATION: Select the CareerNet datasets you want to process.
# Options available: 'general', 'health', 'technology'
# ==============================================================================
SELECTED_DATASETS = ['general', 'health', 'technology']

careernet_dfs = {}

print("=== DOWNLOADING DATASETS ===")
for ds in SELECTED_DATASETS:
    # Construct the raw GitHub URL using the user's keyword
    url = f'https://raw.githubusercontent.com/RenaissancePhilanthropy/careernet-data/main/Datasets/{ds}_public_v1.1.csv'

    try:
        print(f"Loading '{ds}' dataset from GitHub...")
        df = pd.read_csv(url)
        careernet_dfs[ds] = df
        print(f"✅ Success! ({len(df)} rows loaded)")
    except Exception as e:
        print(f"❌ Error loading '{ds}' dataset: {e}. Please check the spelling or URL.")

if careernet_dfs:
    # Display a preview of the first loaded dataset
    sample_key = list(careernet_dfs.keys())[0]
    print(f"\nPreview of '{sample_key}' dataset:")
    display(careernet_dfs[sample_key].head(2))

## Step 4: Standardize SOC Codes

To merge the datasets, the SOC codes in both DataFrames must match perfectly. BLS formats SOC codes like `11-1011`. CareerNet allows for multiple SOC codes per row, separated by semicolons, and stores the SOC descriptive label alongside the code.

This step parses and explodes the CareerNet SOC codes into individual rows, validates each token against the SOC format (`##-####`), and discards any values that do not match (such as `"No career mentioned in question"`). Each SOC string lists the full hierarchy (major; minor; broad; detailed) for every detailed occupation, so parent codes such as `29-0000` can repeat many times within one row; each code is kept only once per record. Non-matching rows are retained in the output but will have no LMI data attached.

In [ ]:
careernet_exploded_dfs = {}

if 'careernet_dfs' in locals() and careernet_dfs:
    print("=== STANDARDIZING & EXPLODING SOC CODES ===")

    # Standardize BLS SOC codes once
    lmi_df['soc_code'] = lmi_df['soc_code'].astype(str).str.strip()

    # SOC code pattern: two digits, hyphen, four digits (e.g., 11-1011)
    soc_pattern = re.compile(r'^\d{2}-\d{4}$')

    def extract_soc_codes(soc_string):
        if pd.isna(soc_string) or str(soc_string).strip().lower() == 'nan':
            return [np.nan]
        parts = str(soc_string).split(';')
        codes = [part.strip().split(' ')[0] for part in parts
                 if part.strip() and soc_pattern.match(part.strip().split(' ')[0])]
        # A row's SOC string repeats parent codes once per detailed occupation;
        # keep each code once so a record does not get identical duplicate rows
        codes = list(dict.fromkeys(codes))
        return codes if codes else [np.nan]

    for ds_name, df in careernet_dfs.items():
        if 'soc_code' not in df.columns:
            print(f"❌ Error: 'soc_code' column not found in '{ds_name}'. Skipping.")
            continue

        # Add a unique row ID for validation
        df = df.copy()

        if 'row_id' not in df.columns:
            df['row_id'] = df.index

        # Extract codes, explode, and standardize
        df['extracted_soc_list'] = df['soc_code'].apply(extract_soc_codes)
        exploded_df = df.explode('extracted_soc_list')
        exploded_df['extracted_soc_list'] = exploded_df['extracted_soc_list'].apply(
            lambda x: str(x).strip() if pd.notna(x) else np.nan
        )

        careernet_exploded_dfs[ds_name] = exploded_df
        print(f"✅ '{ds_name}': {len(df)} original rows -> {len(exploded_df)} exploded rows.")

## Step 5: Merge LMI Data into CareerNet

Now we join. This is a left join: every exploded CareerNet row (one per SOC code per original record) is kept, and BLS statistics are added wherever the SOC code matches. With **national** data, each row matches at most one BLS record, so the row count does not change. With **state** data, each matched row is repeated once per state in `TARGET_STATES`. For example, WA and AR turn 32,964 exploded `general` rows into 47,075. Rows with no match stay as a single row with empty LMI columns. Every original CareerNet record is still present; Step 7 checks this using `row_id`.

In [ ]:
enriched_careernet_dfs = {}

if 'careernet_exploded_dfs' in locals() and careernet_exploded_dfs:
    print("=== MERGING WITH BLS LMI DATA ===")

    for ds_name, exploded_df in careernet_exploded_dfs.items():
        # Merge the exploded dataframe with the BLS LMI data
        enriched_df = pd.merge(
            exploded_df,
            lmi_df,
            left_on='extracted_soc_list',
            right_on='soc_code',
            how='left'
        )

        # Both inputs have a soc_code column, so pandas suffixes them after merge;
        # rename to descriptive names and drop the redundant BLS copy
        enriched_df.rename(columns={'soc_code_x': 'original_soc_string', 'extracted_soc_list': 'matched_soc_code'}, inplace=True)
        enriched_df.drop(columns=['soc_code_y'], inplace=True)

        enriched_careernet_dfs[ds_name] = enriched_df
        print(f"✅ '{ds_name}' successfully merged!")

    # Show a preview of the first successfully merged dataset
    if enriched_careernet_dfs:
        sample_key = list(enriched_careernet_dfs.keys())[0]
        print(f"\nPreview of '{sample_key}' enriched dataset:")
        display(enriched_careernet_dfs[sample_key].head(2))

## Step 6: Export the Enriched Dataset

Export the augmented dataset to a new CSV file so it can be used for modeling, analytics, or further application development.

*Note: The output filename is auto-generated as `{dataset}_{level}_careernet_lmi-{year}.csv` using the configuration variables set in Step 1. To use a custom name, update the `output_filename` f-string directly.*

*Running in Google Colab?* The CSVs are saved to `/content`. Download them from the **Files** panel (folder icon in the left sidebar) before the runtime resets, or they will be lost.

In [ ]:
if 'enriched_careernet_dfs' in locals() and enriched_careernet_dfs:
    print("=== EXPORTING DATASETS ===")

    # Extract the 4-digit year from BLS_YEAR_SUFFIX (e.g., 'M2025_dl.xlsx' -> '2025')
    year_match = re.search(r'\d{4}', BLS_YEAR_SUFFIX)
    bls_year = year_match.group(0) if year_match else 'unknown-year'

    for ds_name, enriched_df in enriched_careernet_dfs.items():
        # Dynamically create the filename based on the dataset name and BLS year
        output_filename = f'{ds_name}-careernet_{BLS_DATA_LEVEL}-lmi-{bls_year}.csv'

        enriched_df.to_csv(output_filename, index=False)
        print(f"💾 Successfully saved: {output_filename}")

## Step 7: Validation and Quality Assurance

Confirm the merge behaved exactly as expected. Checks that no original CareerNet records were lost and that all expected LMI columns are present in the enriched dataset.

In [ ]:
if 'enriched_careernet_dfs' in locals() and 'careernet_dfs' in locals():
    print("=== VALIDATION CHECK ===")

    # Dynamically get the list of expected new columns
    expected_lmi_cols = [new_col for old_col, new_col in LMI_COLUMN_MAPPING.items() if old_col != 'OCC_CODE']
    new_numeric_cols = [LMI_COLUMN_MAPPING[col] for col in NUMERIC_BLS_COLS if col in LMI_COLUMN_MAPPING]

    for ds_name in enriched_careernet_dfs.keys():
        print(f"\n--- Validating '{ds_name}' ---")
        original_df = careernet_dfs[ds_name]
        enriched_df = enriched_careernet_dfs[ds_name]

        # 1. Preservation Check
        original_row_count = len(original_df)
        unique_original_rows = enriched_df['row_id'].nunique()

        if unique_original_rows == original_row_count:
             print(f"✅ Rows Preserved: {original_row_count} unique records strictly accounted for.")
        else:
             print(f"❌ FAIL: Expected {original_row_count} unique rows, found {unique_original_rows}.")

        # 2. LMI Column Check
        missing_cols = [col for col in expected_lmi_cols if col not in enriched_df.columns]
        if not missing_cols:
            check_col = new_numeric_cols[0] if new_numeric_cols else expected_lmi_cols[0]
            matched_records = enriched_df[check_col].notna().sum()
            total_records = len(enriched_df)
            print(f"✅ Data Mapped: Successfully attached BLS stats to {matched_records} out of {total_records} expanded rows.")
        else:
            print(f"❌ FAIL: Missing LMI columns: {missing_cols}")